In [ ]:
# •	Purpose: Allows the script to accept command-line arguments.
# •	Why?: This enables users to pass parameters to the script dynamically instead of hardcoding them.
'''
parser = argparse.ArgumentParser()
parser.add_argument("query_text", type=str, help="The query text.")
args = parser.parse_args()
print(args.query_text)  # Prints the input from the command line
'''
import argparse

import os

# •	Purpose: Provides functions for file operations like copying, deleting, and moving files or directories.
# •	Why?: Used here to delete the existing Chroma database before creating a new one.
'''
shutil.rmtree("chroma")  # Deletes the "chroma" directory and all its contents
'''
import shutil


from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

# •	Purpose: Represents text data with metadata.
# •	Why?: Instead of working with raw text, we organize it into structured documents.
from langchain.schema.document import Document

# from <module_name> import <function_name>
# •	get_embedding_function.py is a Python file (module) that must be in the same directory or a location in the Python path.
# •	get_embedding_function is the function being imported from that file.
from get_embedding_function import get_embedding_function

from langchain.vectorstores.chroma import Chroma

In [3]:
CHROMA_PATH = "chroma"
DATA_PATH = "data"

In [4]:
def load_documents():
    
    document_loader = PyPDFDirectoryLoader(DATA_PATH)
    
    return document_loader.load()

In [4]:
document_loader = PyPDFDirectoryLoader(DATA_PATH)
doc = document_loader.load()
print(doc)
print(len(doc)) # 12 -> 12 pages of PDFs
doc[0].page_content # This is the first page of the PDF document.

[Document(metadata={'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in', 'creator': 'Adobe Acrobat 7.0', 'creationdate': '2007-05-03T12:38:10-04:00', 'moddate': '2007-05-03T12:52:41-04:00', 'source': 'data/monopoly.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play money and a Banker\'s tray. \nNow there\'s a faster way to play MONOPOLY. Choose to play by \nthe classic rules for buying, renting and selling properties or use the \nSpeed Die to get into the action faster. If you\'ve never played the classic \nMONOPOLY game, refer to the Classic Rules beginning on the next page. \nIf you already know how to play and want to use the Speed Die, just \nread the section below for the additional Speed Die rules. \nSPEED DIE RULES \nLearnins how to Play with the S~eed Die IS a

'MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play money and a Banker\'s tray. \nNow there\'s a faster way to play MONOPOLY. Choose to play by \nthe classic rules for buying, renting and selling properties or use the \nSpeed Die to get into the action faster. If you\'ve never played the classic \nMONOPOLY game, refer to the Classic Rules beginning on the next page. \nIf you already know how to play and want to use the Speed Die, just \nread the section below for the additional Speed Die rules. \nSPEED DIE RULES \nLearnins how to Play with the S~eed Die IS as \n/ \nfast as playing with i\'t. \n1. When starting the game, hand out an extra $1,000 to each player \n(two $5005 should work). The game moves fast and you\'ll need \nthe extra cash to buy and build. \n2. Do not use the Speed Die until you\'ve landed on or passed over \nGO for the 

In [5]:
def split_documents(documents: list[Document]):
    '''
    ✅ Increase chunk_size if:
        •	You need full sentences or paragraphs in a single chunk.
        •	Your document is highly structured (e.g., books, research papers).
        •	The chunks are not retrieving enough relevant content.

    ✅ Decrease chunk_size if:
        •	You need more granular retrieval (e.g., FAQs, short articles).
        •	The LLM has token limitations (some models struggle with long inputs).
        •	The dataset contains many small, independent pieces of text.

    ✅ Increase chunk_overlap if:
        •	Chunks lack continuity when retrieving text.
        •	The model struggles to answer questions due to missing context.
        •	You have highly technical or structured data (e.g., legal, medical).

    ✅ Decrease chunk_overlap if:
        •	Chunks contain too much duplicate text, leading to wasted token space.
        •	The dataset is very large, and reducing redundancy improves storage efficiency
    '''  

    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800, # 800 characters
        chunk_overlap = 80, # 80 characters
        length_function = len, # Use the built-in len function to get the length of a string
        # If is_separator_regex = False, it means no regex will be used, and the splitting will be based purely on the provided chunk_size and chunk_overlap parameters.
        is_separator_regex = False, # Don't use a separator regex
        # If is_separator_regex = True, a regex pattern will be used to split the text at specific points defined by the pattern.
    )
    
    # If you want to split based on a specific character, you can set the separator attribute to that character.
    # Split based on a period (.)
    #text_splitter.separator = r"\."
    # Define the regex pattern for paragraph breaks (two newlines or more)
    #text_splitter.separator = r"\n\n"  # Match double newlines (i.e., paragraph breaks) 
    
    # Calling the split_documents method of the instance of RecursiveCharacterTextSplitter which is text_splitter
    return text_splitter.split_documents(documents)

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False,
    )

chunks = text_splitter.split_documents(doc)

print(chunks[0].page_content)
print()
print(chunks[1].page_content)
print()
print(chunks[2].page_content)
print()
print(chunks[3].page_content)

print(len(chunks))
chunks

MONOPOLY 
Property Trading Game from Parker Brothers" 
AGES 8+ 
2 to 8 Players 
Contents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance 
and Community Chest cards, Title Deed cards, play money and a Banker's tray. 
Now there's a faster way to play MONOPOLY. Choose to play by 
the classic rules for buying, renting and selling properties or use the 
Speed Die to get into the action faster. If you've never played the classic 
MONOPOLY game, refer to the Classic Rules beginning on the next page. 
If you already know how to play and want to use the Speed Die, just 
read the section below for the additional Speed Die rules. 
SPEED DIE RULES 
Learnins how to Play with the S~eed Die IS as 
/ 
fast as playing with i't. 
1. When starting the game, hand out an extra $1,000 to each player

1. When starting the game, hand out an extra $1,000 to each player 
(two $5005 should work). The game moves fast and you'll need 
the extra cash to buy and build. 
2. Do not use the Speed Die until you

[Document(metadata={'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in', 'creator': 'Adobe Acrobat 7.0', 'creationdate': '2007-05-03T12:38:10-04:00', 'moddate': '2007-05-03T12:52:41-04:00', 'source': 'data/monopoly.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play money and a Banker\'s tray. \nNow there\'s a faster way to play MONOPOLY. Choose to play by \nthe classic rules for buying, renting and selling properties or use the \nSpeed Die to get into the action faster. If you\'ve never played the classic \nMONOPOLY game, refer to the Classic Rules beginning on the next page. \nIf you already know how to play and want to use the Speed Die, just \nread the section below for the additional Speed Die rules. \nSPEED DIE RULES \nLearnins how to Play with the S~eed Die IS a

In [6]:
# This will create IDs like "data/monopoly.pdf:6:2"
#                           "Page Source : Page Number : Chunk Index"
# Assign a unique ID to each chunk of text, based on its source (e.g., PDF or document), page number, and a chunk index (i.e., position on that page). 
# Keep track of where the chunk came from in the original document, especially when splitting the text into smaller chunks for downstream tasks like indexing, retrieval, or further processing.
def calculate_chunk_ids(chunks):

    last_page_id = None
    current_chunk_index = 0

    # Apply the chunk ID calculation to each chunk.
    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"

        # If the page ID is the same as the last one (same source and page), increment the index.
        if current_page_id == last_page_id:
            # If yes, it increments the chunk index (current_chunk_index) to distinguish between multiple chunks on the same page.
            current_chunk_index += 1
        else:
            # If no (i.e., the page has changed), it resets the chunk index back to 0.
            current_chunk_index = 0

        # Calculate the chunk ID.
        # This generates a unique chunk_id by concatenating the current_page_id and the current_chunk_index. 
        # The resulting ID is in the format source:page:chunk_index. 
        # For example, if the chunk is from page 3 of document.pdf and it’s the second chunk on that page, the chunk_id would be "document.pdf:3:1".
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id

        # Add id section with chunk_id to the page meta-data.
        chunk.metadata["id"] = chunk_id

    return chunks

In [8]:
source = chunks[12].metadata.get("source") # data/monopoly.pdf
page = chunks[12].metadata.get("page") # 4

print(source)
print(page)

current_page_id = f"{source}:{page}"
print(current_page_id) # data/monopoly.pdf:4

last_page_id = None
current_chunk_index = 0

if current_page_id == last_page_id:
    current_chunk_index += 1
else:
    current_chunk_index = 0
    
current_chunk_index # 0

chunk_id = f"{current_page_id}:{current_chunk_index}"
last_page_id = current_page_id

chunk_id # data/monopoly.pdf:4:0
last_page_id # data/monopoly.pdf:4
current_page_id # data/monopoly.pdf:4

chunks[12].metadata
'''
{'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in',
 'creator': 'Adobe Acrobat 7.0',
 'creationdate': '2007-05-03T12:38:10-04:00',
 'moddate': '2007-05-03T12:52:41-04:00',
 'source': 'data/monopoly.pdf',
 'total_pages': 8,
 'page': 4,
 'page_label': '5'}
✅ 
'''

chunks[12].metadata["id"] = chunk_id
'''
{'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in',
 'creator': 'Adobe Acrobat 7.0',
 'creationdate': '2007-05-03T12:38:10-04:00',
 'moddate': '2007-05-03T12:52:41-04:00',
 'source': 'data/monopoly.pdf',
 'total_pages': 8,
 'page': 4,
 'page_label': '5',
✅'id': 'data/monopoly.pdf:4:0'}
'''

data/monopoly.pdf
4
data/monopoly.pdf:4


"\n{'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in',\n 'creator': 'Adobe Acrobat 7.0',\n 'creationdate': '2007-05-03T12:38:10-04:00',\n 'moddate': '2007-05-03T12:52:41-04:00',\n 'source': 'data/monopoly.pdf',\n 'total_pages': 8,\n 'page': 4,\n 'page_label': '5',\n✅'id': 'data/monopoly.pdf:4:0'}\n"

In [7]:
# This function deletes the entire Chroma database directory if it exists.
def clear_database():
    # Checks if the folder (Chroma database) exists.
    if os.path.exists(CHROMA_PATH):
        # Checks if the folder (Chroma database) exists.
        # shutil.rmtree() stands for remove tree, meaning it deletes an entire directory tree recursively.
        shutil.rmtree(CHROMA_PATH)

In [8]:
def add_to_chroma(chunks: list[Document]):
    
    # Load the existing database.
    db = Chroma(
        # Loads an existing Chroma database from the directory CHROMA_PATH.
        persist_directory = CHROMA_PATH, 
        # Convert text chunks into vector embeddings before adding them to the database
        # Call the get_embedding_function().py to get the embedding function.
        embedding_function = get_embedding_function()
    )

    # Calls calculate_chunk_ids(chunks), a separate function that assigns unique IDs to each chunk.
    # "data/monopoly.pdf:6:2" (Source : Page Number : Chunk Index)
    # This prevents duplicate chunks from being added.
    chunks_with_ids = calculate_chunk_ids(chunks)

    # Fetches all stored documents but only includes their IDs (no embeddings or metadata).
    # IDs are always included by default
    existing_items = db.get(include=[])  
	# existing_items["ids"] returns a list of document IDs in the database.
	# set(existing_items["ids"]) converts this list into a set for fast lookups.  
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    # This list will store only the new documents (chunks) that are not already in the database.
	# The database should only store unique documents to avoid unnecessary duplicates.
    new_chunks = []
    # new_chunks = [chunk for chunk in chunks_with_ids if chunk.metadata["id"] not in existing_ids]
    for chunk in chunks_with_ids:
        # existing_ids is a set that contains all document IDs already stored in the database.
        # If the chunk’s ID is not found in existing_ids, it means this chunk is new, and we add it to new_chunks.
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    # Check if there are any new documents to add.
    # If new_chunks is empty, that means all the incoming chunks already existed in the database.
	# If it contains elements, then there are new documents to be added.
    if len(new_chunks):
        print(f"👉 Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        '''
        new_chunk_ids = []
        for chunk in new_chunks:
            new_chunk_ids.append(chunk.metadata["id"])
        '''
        db.add_documents(new_chunks, ids = new_chunk_ids)
        # Ensures that the added documents remain stored permanently on disk.
        # ChromaDB automatically persists changes now, but this command is kept for safety.
        db.persist()
    else:
        print("✅ No new documents to add")

In [9]:
chunks[0].metadata
'''
{'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in',
 'creator': 'Adobe Acrobat 7.0',
 'creationdate': '2007-05-03T12:38:10-04:00',
 'moddate': '2007-05-03T12:52:41-04:00',
 'source': 'data/monopoly.pdf',
 'total_pages': 8,
 'page': 0,
 'page_label': '1'}
'''

chunks_with_ids = calculate_chunk_ids(chunks)
chunks_with_ids[0].metadata
'''
{'producer': 'Adobe Acrobat 7.0 Paper Capture Plug-in',
 'creator': 'Adobe Acrobat 7.0',
 'creationdate': '2007-05-03T12:38:10-04:00',
 'moddate': '2007-05-03T12:52:41-04:00',
 'source': 'data/monopoly.pdf',
 'total_pages': 8,
 'page': 0,
 'page_label': '1',
 'id': 'data/monopoly.pdf:0:0'}
'''
from langchain_community.embeddings.ollama import OllamaEmbeddings
#from langchain_community.embeddings.bedrock import BedrockEmbeddings


def get_embedding_function():
    
    #embeddings = BedrockEmbeddings(
    #    credentials_profile_name = "default", 
    #    region_name = "us-east-1"
    #)
    
    embeddings = OllamaEmbeddings(model = "nomic-embed-text")
    
    return embeddings

db = Chroma(persist_directory=CHROMA_PATH, embedding_function=get_embedding_function())

/var/folders/dh/551p960n6mv9t9byzj8pp0640000gp/T/ipykernel_90899/1870423944.py:37: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model = "nomic-embed-text")
/var/folders/dh/551p960n6mv9t9byzj8pp0640000gp/T/ipykernel_90899/1870423944.py:41: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(persist_directory=CHROMA_PATH, embedding_function=get_embeddin

In [11]:
existing_items = db.get(include=[])  
existing_ids = set(existing_items["ids"])
print(f"Number of existing documents in DB: {len(existing_ids)}")

existing_items

Number of existing documents in DB: 0


{'ids': [],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'data': None,
 'metadatas': None,
 'included': []}

In [ ]:
def main():

    # Check if the database should be cleared (using the --clear flag).
    parser = argparse.ArgumentParser()
    # Defines an optional flag --reset.
	# action="store_true" means:
	# If present in the command (e.g., python3 populate_database.py --reset), args.reset will be True.
	# If not present, args.reset will be False.
	# help="Reset the database." provides a description if the user runs python3 populate_database.py --help.
    parser.add_argument("--reset", action="store_true", help="Reset the database.")
    args = parser.parse_args()
    # If --reset is used (args.reset == True)
    if args.reset:
        
        print("✨ Clearing Database")
        
        clear_database()

    # Create (or update) the data store.
    documents = load_documents()
    chunks = split_documents(documents)
    add_to_chroma(chunks)

In [11]:
if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--reset]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/BAEK/Library/Jupyter/runtime/kernel-v320d1b58aa41319d8e00ac5e9b20f1d6e9dcafb08.json


SystemExit: 2

/Users/BAEK/Code/python/llm/rag-tutorial-v2/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3554: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
